[English](https://colab.research.google.com/github/kanoyo-git/NVC/blob/main/NVC.ipynb) | [Русский](https://colab.research.google.com/github/kanoyo-git/NVC/blob/main/NVC.ru.ipynb)

# NVC в Google Colab

NVC — локальный инструмент на основе RVC для конвертации голоса и обучения моделей. Он поддерживает обработку отдельных файлов и папок, разделение вокала и динамический автотюн по профилю датасета.

Этот блокнот создаёт изолированное окружение Python 3.12, загружает необходимые ресурсы v2 и запускает NVC Studio.

Выберите среду выполнения с GPU и последовательно запустите ячейки 0–9. По умолчанию для удалённого доступа используется LocalTunnel.

**Дотренировка.** Включите `USE_DRIVE_TRAINING` в ячейке 1: эксперименты из `logs` (последние чекпоинты G/D, признаки, индекс) и обученные модели копируются на Google Drive, пока работает Studio, а в новой сессии их восстанавливает ячейка 7. Затем откройте в Studio *Обучение → Шаг 4 · Дотренировка*.

**TensorBoard** доступен на вкладке *TensorBoard* в Studio и, по желанию, прямо в Colab (ячейка 8).


In [ ]:
# @title 0 · Проверка GPU
import os, shutil, subprocess, sys
assert shutil.which("nvidia-smi"), "Среда выполнения -> Сменить среду выполнения -> GPU"
subprocess.run(["nvidia-smi"], check=True)
print("python:", sys.version.split()[0])
print("CUDA доступна:", os.environ.get("CUDA_VISIBLE_DEVICES", "<unset>"))


In [ ]:
# @title 1 · Настройки
from pathlib import Path

#@markdown #### Репозиторий
NVC_GIT_URL = "https://github.com/kanoyo-git/NVC.git"  # @param {type:"string"}
NVC_GIT_REF = "main"  # @param ["main", "experimental"]

#@markdown #### Пути
PROJECT_DIR = "/content/NVC"  # @param {type:"string"}
VENV_DIR = "/content/nvc-venv"  # @param {type:"string"}
DRIVE_CACHE_DIR = "/content/drive/MyDrive/NVC-cache"  # @param {type:"string"}
DRIVE_MODELS_PATH = "Models"  # @param {type:"string"}

#@markdown #### Переключатели
USE_DRIVE_CACHE = False  # @param {type:"boolean"}
USE_DRIVE_MODELS = False  # @param {type:"boolean"}

#@markdown #### Интерфейс
INTERFACE = "studio"  # @param ["studio", "legacy"]
ACCESS = "localtunnel"  # @param ["colab_iframe", "colab_window", "cloudflared", "localtunnel", "bore"]
GUI_PORT = 7865  # @param {type:"integer"}

#@markdown #### Обучение
# USE_DRIVE_TRAINING хранит эксперименты на Google Drive, чтобы дотренировать модель в новой сессии.
# EXTRA_PRETRAIN заранее скачивает один претрейн сообщества (RVC v2 скачивается всегда); остальные — на вкладке «Обучение».
USE_DRIVE_TRAINING = False  # @param {type:"boolean"}
DRIVE_TRAINING_PATH = "NVC-training"  # @param {type:"string"}
DRIVE_SYNC_MINUTES = 10  # @param {type:"integer"}
EXTRA_PRETRAIN = "none"  # @param ["none", "Beatz Forge", "IMA V1", "Italia", "KLM4", "Nanashi v1.7", "Ov2Super", "Rigel Base", "Rigel FineTuned", "RIN_E3", "SingerPreTrain", "Snowie", "Snowie-X-RinE3", "Titan", "UKA"]
EXTRA_PRETRAIN_SR = "40k"  # @param ["32k", "40k", "48k"]

PROJECT = Path(PROJECT_DIR)
VENV = Path(VENV_DIR)
PY = VENV / "bin" / "python"
UV_BIN = None
print("ветка:", NVC_GIT_REF)
print("интерфейс:", INTERFACE)
print("доступ:", ACCESS)
print("обучение на Drive:", USE_DRIVE_TRAINING)
print("доп. претрейн:", EXTRA_PRETRAIN, EXTRA_PRETRAIN_SR)


In [ ]:
# @title 2 · Системные пакеты и uv
import os, shutil, subprocess
from pathlib import Path

def sh(cmd, **kwargs):
    print("+", cmd if isinstance(cmd, str) else " ".join(map(str, cmd)))
    subprocess.run(cmd, check=True, **kwargs)

def resolve_uv():
    extra = [Path.home()/".local"/"bin", Path("/root/.local/bin"), Path.home()/".cargo"/"bin", Path("/usr/local/bin")]
    os.environ["PATH"] = os.pathsep.join(str(p) for p in extra if p.is_dir()) + os.pathsep + os.environ["PATH"]
    found = shutil.which("uv")
    if found:
        return Path(found)
    for folder in extra:
        cand = folder / "uv"
        if cand.is_file():
            return cand
    return None

sh(["apt-get", "update", "-qq"])
sh(["apt-get", "install", "-y", "-qq", "ffmpeg", "libsndfile1", "libportaudio2", "git", "unzip", "curl"])
UV_BIN = resolve_uv()
if UV_BIN is None:
    install_dir = Path.home()/".local"/"bin"
    install_dir.mkdir(parents=True, exist_ok=True)
    env = os.environ.copy()
    env["UV_INSTALL_DIR"] = str(install_dir)
    env["UV_NO_MODIFY_PATH"] = "1"
    sh("curl -LsSf https://astral.sh/uv/install.sh | sh", shell=True, env=env)
    UV_BIN = resolve_uv()
if UV_BIN is None:
    sh(["python3", "-m", "pip", "install", "-q", "uv"])
    UV_BIN = resolve_uv()
if UV_BIN is None:
    raise FileNotFoundError("исполняемый файл uv не найден")
os.environ["PATH"] = str(UV_BIN.parent) + os.pathsep + os.environ["PATH"]
sh([str(UV_BIN), "--version"])
print("uv:", UV_BIN)


In [ ]:
# @title 3 · Получение / обновление проекта
import os, subprocess
from pathlib import Path

def sh(cmd, **kwargs):
    print("+", cmd if isinstance(cmd, str) else " ".join(map(str, cmd)))
    subprocess.run(cmd, check=True, **kwargs)

if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    Path(DRIVE_CACHE_DIR).mkdir(parents=True, exist_ok=True)

if (PROJECT / ".git").is_dir():
    sh(["git", "-C", str(PROJECT), "fetch", "--depth", "1", "origin", NVC_GIT_REF])
    sh(["git", "-C", str(PROJECT), "checkout", "-f", "FETCH_HEAD"])
    print("проект обновлён:", PROJECT)
elif (PROJECT / "gui.py").is_file():
    print("используется существующий проект:", PROJECT)
elif NVC_GIT_URL.strip():
    if PROJECT.exists():
        sh(["rm", "-rf", str(PROJECT)])
    sh(["git", "clone", "--depth", "1", "--branch", NVC_GIT_REF, NVC_GIT_URL.strip(), str(PROJECT)])
else:
    raise SystemExit("Укажите NVC_GIT_URL или поместите gui.py в PROJECT_DIR")

os.chdir(PROJECT)
print("рабочий каталог:", Path.cwd())


In [ ]:
# @title 4 · Окружение Python
import os, subprocess
from pathlib import Path

def sh(cmd, **kwargs):
    print("+", cmd if isinstance(cmd, str) else " ".join(map(str, cmd)))
    subprocess.run(cmd, check=True, **kwargs)

os.chdir(PROJECT)
req_src = PROJECT / "requirments_cu128_py312.txt"
req_clean = Path("/tmp/nvc-cu128-pypi.txt")
lines = [ln for ln in req_src.read_text(encoding="utf-8").splitlines() if not ln.strip().startswith(("--index-url", "--extra-index-url"))]
req_clean.write_text("\n".join(lines) + "\n", encoding="utf-8")
if not PY.is_file():
    sh([str(UV_BIN), "venv", str(VENV), "--python", "3.12", "--clear"])
sh([str(UV_BIN), "pip", "install", "--python", str(PY), "torch==2.7.1+cu128", "torchaudio==2.7.1+cu128", "--index-url", "https://download.pytorch.org/whl/cu128"])
sh([str(UV_BIN), "pip", "install", "--python", str(PY), "-r", str(req_clean), "huggingface_hub>=0.25,<1", "--index-url", "https://pypi.org/simple"])
sh([str(UV_BIN), "pip", "install", "--python", str(PY), "noisereduce>=2.0.0", "soxr>=0.3.0", "--index-url", "https://pypi.org/simple"])
sh([str(PY), "-c", "import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda, 'ok', torch.cuda.is_available())"])


In [ ]:
# @title 5 · Загрузка ресурсов v2
import codecs, os, shutil, zipfile
from pathlib import Path
from huggingface_hub import hf_hub_download, snapshot_download

os.chdir(PROJECT)
if USE_DRIVE_CACHE:
    hf_home = Path(DRIVE_CACHE_DIR) / "huggingface"
    hf_home.mkdir(parents=True, exist_ok=True)
    os.environ["HF_HOME"] = str(hf_home)

MODEL_REPO = codecs.decode("yw1995/IbvprPbairefvbaJroHV", "rot_13")
NMVPE_REPO = "kanoyo/nvc_resources"
MHUBERT_REPO = "utter-project/mHuBERT-147"
for rel in ("assets/hubert_base", "assets/mhubert147", "assets/rmvpe", "assets/nmvpe", "assets/pretrained_v2", "assets/pymss_weights", "assets/weights", "assets/indices", "logs", ".model-downloads"):
    (PROJECT / rel).mkdir(parents=True, exist_ok=True)

snapshot_download(MODEL_REPO, revision="main", allow_patterns=["hubert_base/*"], local_dir=str(PROJECT / "assets"))
snapshot_download(MHUBERT_REPO, revision="main", allow_patterns=["config.json", "model.safetensors", "preprocessor_config.json"], local_dir=str(PROJECT / "assets" / "mhubert147"))
hf_hub_download(MODEL_REPO, "rmvpe.pt", revision="main", local_dir=str(PROJECT / "assets" / "rmvpe"))
nmvpe_source = Path(hf_hub_download(NMVPE_REPO, "extractors/nmvpe.pt", revision="main"))
nmvpe_target = PROJECT / "assets" / "nmvpe" / "nmvpe.pt"
if not nmvpe_target.is_file() or nmvpe_target.stat().st_size != nmvpe_source.stat().st_size:
    shutil.copy2(nmvpe_source, nmvpe_target)
snapshot_download(MODEL_REPO, revision="main", allow_patterns=["pretrained_v2/*"], local_dir=str(PROJECT / "assets"))
mute_zip = hf_hub_download(MODEL_REPO, "mute.zip", revision="main", local_dir=str(PROJECT / ".model-downloads"))
snapshot_download(MODEL_REPO, revision="main", allow_patterns=["pymss_weights/*"], local_dir=str(PROJECT / "assets"))
with zipfile.ZipFile(mute_zip) as archive:
    archive.extractall(PROJECT / "logs")

# Заранее загружаем альтернативные эмбеддеры ContentVec / SPIN / SPIN-v2, чтобы первый
# запуск инференса или обучения не зависел от сети. hubert_base загружается выше.
# Веса зеркалируются из IAHispano/Applio по лицензии MIT; NVC обычно загружает их
# при первом использовании через infer/hubert.py, а здесь мы заранее заполняем кэш.
import urllib.request
EMBEDDER_BASE = "https://huggingface.co/IAHispano/Applio/resolve/main/Resources/embedders"
for embedder in ("contentvec", "spin", "spin-v2"):
    embedder_dir = PROJECT / "assets" / embedder
    embedder_dir.mkdir(parents=True, exist_ok=True)
    for file_name in ("pytorch_model.bin", "config.json"):
        target = embedder_dir / file_name
        if target.is_file():
            continue
        url = f"{EMBEDDER_BASE}/{embedder}/{file_name}"
        print(f"загрузка {embedder}/{file_name} ...")
        with urllib.request.urlopen(url) as response, open(target, "wb") as out:
            while True:
                chunk = response.read(1 << 20)
                if not chunk:
                    break
                out.write(chunk)

required = [
    PROJECT/"assets/hubert_base/config.json",
    PROJECT/"assets/hubert_base/preprocessor_config.json",
    PROJECT/"assets/hubert_base/pytorch_model.bin",
    PROJECT/"assets/rmvpe/rmvpe.pt",
    PROJECT/"assets/nmvpe/nmvpe.pt",
    PROJECT/"assets/pretrained_v2/f0G40k.pth",
    PROJECT/"assets/pretrained_v2/f0D40k.pth",
    PROJECT/"assets/contentvec/pytorch_model.bin",
    PROJECT/"assets/contentvec/config.json",
    PROJECT/"assets/mhubert147/model.safetensors",
    PROJECT/"assets/mhubert147/config.json",
    PROJECT/"assets/mhubert147/preprocessor_config.json",
    PROJECT/"logs/mute",
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("отсутствуют ресурсы:\n" + "\n".join(missing))
# Необязательные модели 48 кГц для более качественного обучения; версия 40 кГц доступна всегда.
pretrained_48k = [
    PROJECT/"assets/pretrained_v2/f0G48k.pth",
    PROJECT/"assets/pretrained_v2/f0D48k.pth",
]
missing_48k = [str(p) for p in pretrained_48k if not p.exists()]
if missing_48k:
    print("примечание: модели 48 кГц не найдены и будут загружены при необходимости:", missing_48k)
else:
    print("модели 48 кГц готовы")
print("ресурсы v2 готовы")

if EXTRA_PRETRAIN != "none":
    import subprocess, sys
    catalog_script = PROJECT / "tools/pretrained_catalog.py"
    if not catalog_script.is_file():
        print(f"внимание: {catalog_script} не найден: в ветке {NVC_GIT_REF!r} нет каталога претрейнов, {EXTRA_PRETRAIN} пропущен. Поставьте NVC_GIT_REF = main и перезапустите ячейку 3.")
    else:
        # Stream the child's output so download progress and errors show up in the cell.
        proc = subprocess.Popen(
            [str(PY), str(catalog_script), "download", EXTRA_PRETRAIN, EXTRA_PRETRAIN_SR],
            cwd=str(PROJECT), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        )
        while True:
            chunk = proc.stdout.read1(4096)
            if not chunk:
                break
            sys.stdout.write(chunk.decode("utf-8", "replace"))
            sys.stdout.flush()
        if proc.wait() != 0:
            print(f"внимание: {EXTRA_PRETRAIN} {EXTRA_PRETRAIN_SR} не скачан (код {proc.returncode}, подробности в логе выше). Обучение работает и со стандартным претрейном RVC v2; скачать набор можно повторно на вкладке «Обучение».")
        else:
            print(f"{EXTRA_PRETRAIN} {EXTRA_PRETRAIN_SR} готов")


In [ ]:
# @title 6 · Голосовые модели с Google Drive
# Включите тумблер в настройках и запустите ячейку. Она монтирует Drive,
# разрешает путь DRIVE_MODELS_PATH (относительно MyDrive или абсолютный) и
# раскладывает симлинки на модели в assets/weights и assets/indices — ничего
# не копируется. Ожидаемый формат:
#   <DRIVE_MODELS_PATH>/<Имя_Модели>/<любое-имя>.pth + <любое-имя>.index
# Имя .index-файла не обязано совпадать с именем модели.
import os
from pathlib import Path

if not USE_DRIVE_MODELS:
    print("отключено (USE_DRIVE_MODELS = False)")
else:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive", force_remount=False)

    my_drive = Path("/content/drive/MyDrive")
    raw_path = str(DRIVE_MODELS_PATH).strip().strip('"').rstrip("/")
    if not raw_path:
        raise ValueError("DRIVE_MODELS_PATH is empty")
    given = Path(raw_path)
    models_root = given if given.is_absolute() else my_drive / given
    if not models_root.is_dir():
        raise RuntimeError(
            "папка с моделями не найдена: %s. Проверьте DRIVE_MODELS_PATH в настройках "
            "(относительные пути считаются от MyDrive); если папка общая, добавьте "
            "ярлык в MyDrive («Организовать» -> «Добавить ярлык») с тем же именем." % models_root
        )

    weights_root = PROJECT / "assets/weights"
    indices_root = PROJECT / "assets/indices"
    weights_root.mkdir(parents=True, exist_ok=True)
    indices_root.mkdir(parents=True, exist_ok=True)

    linked, kept, summary = [], [], {}
    for source in sorted(models_root.rglob("*")):
        suffix = source.suffix.lower()
        if suffix not in (".pth", ".index") or not source.is_file():
            continue
        relative = source.relative_to(models_root)
        if len(relative.parts) >= 2:
            model_name, inner = relative.parts[0], Path(*relative.parts[1:])
        else:
            model_name, inner = source.stem, Path(relative.name)
        target = (weights_root if suffix == ".pth" else indices_root) / model_name / inner
        if target.exists() and not target.is_symlink():
            kept.append(str(target.relative_to(PROJECT)))
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            if target.is_symlink():
                target.unlink()
            target.symlink_to(source.absolute())
            linked.append(str(target.relative_to(PROJECT)))
        counts = summary.setdefault(model_name, [0, 0])
        counts[0 if suffix == ".pth" else 1] += 1

    print("папка моделей:", models_root)
    for name in sorted(summary):
        models_count, indices_count = summary[name]
        print("  %s: %d модель(и), %d индекс(ов)" % (name, models_count, indices_count))
    for item in linked:
        print("симлинк:", item)
    for item in sorted(set(kept)):
        print("оставлен существующий файл:", item)
    if not summary:
        print("в папке нет файлов .pth/.index:", models_root)


In [ ]:
# @title 7 · Эксперименты обучения на Google Drive
# Восстанавливает эксперименты (logs/<имя>) и обученные модели из DRIVE_TRAINING_PATH
# (относительно MyDrive или абсолютный путь) и задаёт синхронизацию для ячейки запуска.
# На Drive хранится только последняя пара чекпоинтов G/D каждого эксперимента.
import os, re, shutil, time
from pathlib import Path

def _copy_if_changed(src, dst):
    if dst.is_symlink():
        return 0
    if dst.is_file():
        a, b = src.stat(), dst.stat()
        if a.st_size == b.st_size and int(a.st_mtime) == int(b.st_mtime):
            return 0
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + ".part")
    shutil.copy2(src, tmp)
    os.replace(tmp, dst)
    return 1

def _latest_pair(folder, settled):
    # Newest step that has both G_<n> and D_<n> fully written; a G/D pair from
    # different steps would make train.py refuse to resume.
    steps = {}
    for path in folder.glob("[GD]_*.pth"):
        if re.fullmatch(r"[GD]_\d+\.pth", path.name) and path.stat().st_mtime <= settled:
            steps.setdefault(int(path.stem[2:]), {})[path.name[0]] = path
    complete = [step for step, pair in steps.items() if len(pair) == 2]
    return set(steps[max(complete)].values()) if complete else set()

def sync_training_to_drive(settle=30):
    # Files written in the last `settle` seconds may still be in progress
    # (torch.save of a checkpoint); the next round picks them up.
    copied, fresh = 0, time.time() - settle
    logs = PROJECT / "logs"
    for exp in sorted(p for p in logs.iterdir() if p.is_dir() and p.name != "mute"):
        target = TRAINING_DRIVE / "logs" / exp.name
        keep = _latest_pair(exp, fresh)
        for src in exp.rglob("*"):
            if not src.is_file() or src.suffix in (".part", ".tmp") or src.stat().st_mtime > fresh:
                continue
            if src.parent == exp and re.fullmatch(r"[GD]_\d+\.pth", src.name) and src not in keep:
                continue
            copied += _copy_if_changed(src, target / src.relative_to(exp))
        if keep and all((target / p.name).is_file() and (target / p.name).stat().st_size == p.stat().st_size for p in keep):
            for old in target.glob("[GD]_*.pth"):
                if old.name not in {p.name for p in keep}:
                    old.unlink()
    for src in (PROJECT / "assets" / "weights").glob("*.pth"):
        if not src.is_symlink() and src.stat().st_mtime <= fresh:
            copied += _copy_if_changed(src, TRAINING_DRIVE / "weights" / src.name)
    return copied

def restore_training_from_drive():
    restored = 0
    for exp in sorted(p for p in (TRAINING_DRIVE / "logs").glob("*") if p.is_dir()):
        for src in exp.rglob("*"):
            if src.is_file() and src.suffix != ".part":
                restored += _copy_if_changed(src, PROJECT / "logs" / exp.name / src.relative_to(exp))
    for src in (TRAINING_DRIVE / "weights").glob("*.pth"):
        restored += _copy_if_changed(src, PROJECT / "assets" / "weights" / src.name)
    return restored

if not USE_DRIVE_TRAINING:
    print("отключено (USE_DRIVE_TRAINING = False)")
else:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive", force_remount=False)
    given = Path(str(DRIVE_TRAINING_PATH).strip().strip('"').rstrip("/") or "NVC-training")
    TRAINING_DRIVE = given if given.is_absolute() else Path("/content/drive/MyDrive") / given
    (TRAINING_DRIVE / "logs").mkdir(parents=True, exist_ok=True)
    (TRAINING_DRIVE / "weights").mkdir(parents=True, exist_ok=True)
    print("восстановлено с Drive: %d файл(ов)" % restore_training_from_drive())
    names = sorted(p.name for p in (TRAINING_DRIVE / "logs").iterdir() if p.is_dir())
    if names:
        print("эксперименты на Drive:", ", ".join(names))
    else:
        print("на Drive пока нет экспериментов:", TRAINING_DRIVE)


In [ ]:
# @title 8 · TensorBoard (необязательно)
# TensorBoard также встроен в Studio (вкладка TensorBoard) и работает при любом ACCESS.
# Эта ячейка показывает его прямо здесь; он обновляется, пока работает ячейка запуска.
SHOW_TENSORBOARD_IN_COLAB = True  # @param {type:"boolean"}
TENSORBOARD_PORT = 6006  # @param {type:"integer"}

if SHOW_TENSORBOARD_IN_COLAB:
    (PROJECT / "logs").mkdir(parents=True, exist_ok=True)
    ipython = get_ipython()
    ipython.run_line_magic("load_ext", "tensorboard")
    ipython.run_line_magic("tensorboard", "--logdir %s --port %d --reload_interval 15" % (PROJECT / "logs", TENSORBOARD_PORT))


In [ ]:
# @title 9 · Запуск
import os, re, shutil, socket, subprocess, sys, threading, time
from pathlib import Path
from google.colab import output

os.chdir(PROJECT)

CHILD_PROCS = []

def port_open(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(0.4)
        return sock.connect_ex(("127.0.0.1", port)) == 0

def pump(stream):
    for line in stream:
        sys.stdout.write(line)
        sys.stdout.flush()

def wait_port(port, proc=None, timeout=180):
    deadline = time.time() + timeout
    while time.time() < deadline:
        if proc is not None and proc.poll() is not None:
            raise RuntimeError("процесс завершился с кодом %s" % proc.returncode)
        if port_open(port):
            return
        time.sleep(0.4)
    raise TimeoutError("порт %s не открылся" % port)

def start_app():
    env = os.environ.copy()
    env["NVC_OFFLINE_CUDA_GRAPH"] = "0"
    env["PYTHONUNBUFFERED"] = "1"
    env["MPLBACKEND"] = "Agg"
    entrypoint = "legacy_gui.py" if INTERFACE == "legacy" else "gui.py"
    cmd = [str(PY), "-u", entrypoint, "--colab", "--noautoopen", "--port", str(GUI_PORT)]
    print("запуск:", " ".join(cmd))
    proc = subprocess.Popen(cmd, cwd=str(PROJECT), env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    CHILD_PROCS.append(proc)
    threading.Thread(target=pump, args=(proc.stdout,), daemon=True).start()
    wait_port(GUI_PORT, proc)
    return proc

def ensure_cloudflared():
    bin_path = Path("/tmp/cloudflared")
    if not bin_path.is_file():
        subprocess.run(["curl", "-L", "--fail", "-o", str(bin_path), "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
        bin_path.chmod(0o755)
    return bin_path

def ensure_bore():
    bin_path = Path("/tmp/bore")
    if not bin_path.is_file():
        archive = Path("/tmp/bore.tar.gz")
        subprocess.run(["curl", "-L", "--fail", "-o", str(archive), "https://github.com/ekzhang/bore/releases/download/v0.6.0/bore-v0.6.0-x86_64-unknown-linux-musl.tar.gz"], check=True)
        subprocess.run(["tar", "-xzf", str(archive), "-C", "/tmp"], check=True)
        bin_path.chmod(0o755)
    return bin_path

def first_url(proc, pattern, timeout=90):
    compiled = re.compile(pattern)
    deadline = time.time() + timeout
    while time.time() < deadline:
        line = proc.stdout.readline()
        if not line:
            if proc.poll() is not None:
                break
            continue
        sys.stdout.write(line)
        match = compiled.search(line)
        if match:
            threading.Thread(target=pump, args=(proc.stdout,), daemon=True).start()
            return match
    threading.Thread(target=pump, args=(proc.stdout,), daemon=True).start()
    raise RuntimeError("туннель не вернул публичный адрес")

def start_cloudflared(port):
    proc = subprocess.Popen([str(ensure_cloudflared()), "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:%s" % port], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    CHILD_PROCS.append(proc)
    match = first_url(proc, r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com")
    return match.group(0)

def start_localtunnel(port):
    if not shutil.which("npx"):
        raise RuntimeError("npx недоступен в этой среде выполнения")
    proc = subprocess.Popen(["npx", "--yes", "localtunnel", "--port", str(port)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    CHILD_PROCS.append(proc)
    match = first_url(proc, r"https://\S+")
    return match.group(0)

def start_bore(port):
    proc = subprocess.Popen([str(ensure_bore()), "local", str(port), "--to", "bore.pub"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    CHILD_PROCS.append(proc)
    match = first_url(proc, r"bore\.pub:(\d+)")
    return "http://bore.pub:%s" % match.group(1)

if USE_DRIVE_TRAINING and "sync_training_to_drive" not in globals():
    raise RuntimeError("сначала запустите ячейку 7 (USE_DRIVE_TRAINING включён)")

if not port_open(GUI_PORT):
    start_app()
else:
    print("повторно используется процесс на порту", GUI_PORT)

if ACCESS == "colab_iframe":
    print("Встроенное окно Colab. Журнал обновляется в реальном времени.")
    output.serve_kernel_port_as_iframe(GUI_PORT, height=920)
elif ACCESS == "colab_window":
    print("Окно Colab.")
    output.serve_kernel_port_as_window(GUI_PORT)
elif ACCESS == "cloudflared":
    print("Cloudflare Quick Tunnel: аккаунт не нужен, но журнал SSE не поддерживается.")
    print(start_cloudflared(GUI_PORT))
elif ACCESS == "localtunnel":
    print("LocalTunnel: аккаунт не нужен.")
    print(start_localtunnel(GUI_PORT))
elif ACCESS == "bore":
    print("Bore: аккаунт не нужен.")
    print(start_bore(GUI_PORT))
else:
    raise ValueError(ACCESS)

if USE_DRIVE_TRAINING:
    def drive_sync_loop():
        while True:
            time.sleep(max(1, DRIVE_SYNC_MINUTES) * 60)
            try:
                copied = sync_training_to_drive()
                if copied:
                    print("[drive] синхронизировано файлов: %d" % copied)
            except Exception as error:
                print("[drive] ошибка синхронизации:", error)

    threading.Thread(target=drive_sync_loop, daemon=True).start()

uptime = time.time()
try:
    while True:
        time.sleep(300)
        minutes = int((time.time() - uptime) // 60)
        state = "открыт" if port_open(GUI_PORT) else "ЗАКРЫТ"
        print("[heartbeat] %d мин. | порт %d %s" % (minutes, GUI_PORT, state))
except KeyboardInterrupt:
    print("\nОстанавливаю сервер и туннели…")
    for p in reversed(CHILD_PROCS):
        try:
            p.terminate()
        except Exception:
            pass
    deadline = time.time() + 5
    for p in CHILD_PROCS:
        try:
            p.wait(timeout=max(0.1, deadline - time.time()))
        except Exception:
            try:
                p.kill()
            except Exception:
                pass
    while port_open(GUI_PORT) and time.time() < deadline:
        time.sleep(0.2)
    if USE_DRIVE_TRAINING:
        print("Финальная синхронизация с Google Drive…")
        try:
            print("[drive] синхронизировано файлов: %d" % sync_training_to_drive(settle=0))
        except Exception as error:
            print("[drive] ошибка синхронизации:", error)
    print("Готово. Можно запускать ячейку заново.")